# Stage 3 — Leak-safe Improved Ensemble

This notebook keeps the team's original architecture:

**LightGBM + CatBoost + Ridge → tuned weighted blend → volatility-aware allocation**.

It uses the existing project modules `src/preprocessing.py`, `src/ensemble.py`, and `src/allocation.py` and fixes the main methodological issues in the previous `improved_ensemble_optuna.ipynb`:

1. imputation is fitted on training rows only;
2. CatBoost feature selection is fitted inside training folds only;
3. model hyperparameters are selected in inner CV;
4. blend weights are selected on inner OOF predictions, not on the outer validation fold;
5. regression metrics use forecasts in target units, while standardised signals are used only for allocation;
6. the volatility-aware risk layer is calibrated on inner OOF data to target a conservative strategy/market volatility ratio;
7. LightGBM tuning enforces a valid `num_leaves <= 2 ** max_depth` relationship and adds explicit regularisation.

The trailing 180-row block has already informed earlier team iterations, so any evaluation on it must be reported as **diagnostic/post-hoc**, not as a fully untouched test set.

In [10]:
# ============================ SETUP ============================
from __future__ import annotations

import json
import pathlib
import sys
import time
from dataclasses import asdict, dataclass
from typing import Any

# Work both when Jupyter starts in the repository root and in model_improvements/.
CWD = pathlib.Path.cwd().resolve()
ROOT = CWD.parent if CWD.name == "model_improvements" else CWD
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

from src import (
    DATE_COL,
    LOOKAHEAD_COLS,
    SEED,
    TARGET,
    ForwardMedianImputer,
    aggregate_folds,
    apply_standardization,
    build_features,
    calibrate_vol_budget_scale,
    causal_standardise,
    evaluate,
    get_folds,
    load_dataset,
    risk_controlled_vol_budget_allocation,
    set_seed,
    spearman_ic,
    standardization_stats,
)
from src.ensemble import (
    blend,
    family_oof_predictions,
    fit_catboost_selector,
    prepare_folds,
    standardize_family_oof,
    tune_blend_weights,
    tune_family,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)
set_seed(SEED)
print(f"Project root: {ROOT}")

Project root: /home/alex/Desktop/PMDL-project/PMDL-Project-Market-Prediction


## 1. Configuration

`QUICK=True` is a smoke test only. Use `QUICK=False` for the full nested-CV experiment that should be compared with the previous ensemble results.

In [11]:
FAMILIES = ("lgbm", "catboost", "ridge")
LAG_ROLL_COLUMNS = [
    "M4", "V13", "S5", "S2", "D2", "E19", "P7", "P6",
    "P3", "P13", "P4", "P5", "M2", "V5",
]

@dataclass(frozen=True)
class RunConfig:
    outer_splits: int = 4
    inner_splits: int = 3
    trials_lgbm: int = 20
    trials_catboost: int = 20
    trials_ridge: int = 15
    trials_blend: int = 100
    catboost_features: int = 120
    selector_iterations: int = 300
    target_vol_ratio: float = 1.05
    target_vol: float = 0.12
    jobs: int = 8

QUICK = False
EVALUATE_DIAGNOSTIC_PUBLIC = False

if QUICK:
    CFG = RunConfig(
        outer_splits=2,
        inner_splits=2,
        trials_lgbm=2,
        trials_catboost=2,
        trials_ridge=2,
        trials_blend=20,
        catboost_features=120,
        selector_iterations=60,
        jobs=4,
    )
    OUTPUT_DIR = ROOT / "results" / "improved_ensemble_smoke"
else:
    CFG = RunConfig(jobs=8)
    OUTPUT_DIR = ROOT / "results" / "improved_ensemble"

print(CFG)
print("Output:", OUTPUT_DIR)

RunConfig(outer_splits=4, inner_splits=3, trials_lgbm=20, trials_catboost=20, trials_ridge=15, trials_blend=100, catboost_features=120, selector_iterations=300, target_vol_ratio=1.05, target_vol=0.12, jobs=8)
Output: /home/alex/Desktop/PMDL-project/PMDL-Project-Market-Prediction/results/improved_ensemble


## 2. Data and feature construction

The feature family is unchanged. The important difference is that learned preprocessing statistics and CatBoost feature selection are no longer fitted globally before validation.

In [12]:
def build_frame() -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    hull = load_dataset()
    lag_cols = [c for c in LAG_ROLL_COLUMNS if c in hull.full.columns]
    feat_df, features = build_features(
        hull.full,
        lag_roll_columns=lag_cols,
        price_features=True,
        cross_terms=True,
    )

    # load_dataset() already reserves the trailing 180 rows as hull.public.
    cut = hull.train[DATE_COL].max()
    train = feat_df.loc[feat_df[DATE_COL] <= cut].reset_index(drop=True)
    public = feat_df.loc[feat_df[DATE_COL] > cut].reset_index(drop=True)

    overlap = set(features) & set(LOOKAHEAD_COLS)
    if overlap:
        raise AssertionError(f"Look-ahead columns in feature list: {sorted(overlap)}")

    return train, public, features

train_df, public_df, FEATURES = build_frame()
print("train:", train_df.shape)
print("diagnostic public:", public_df.shape)
print("features:", len(FEATURES))

train: (7862, 339)
diagnostic public: (180, 339)
features: 335


## 3. Nested-CV helpers

The outer validation fold is evaluation-only. All of the following happen inside the outer training period:

- median/forward-fill fitting;
- CatBoost top-feature selection;
- LightGBM/CatBoost/Ridge hyperparameter tuning;
- family prediction standardisation;
- ensemble-weight tuning;
- risk-scale calibration.

In [13]:
def compact_oof(frame, prepared, family_results):
    mask = np.ones(len(frame), dtype=bool)
    for family in FAMILIES:
        mask &= np.isfinite(family_results[family].raw_predictions)

    row_positions = np.flatnonzero(mask)
    pos_map = {row: i for i, row in enumerate(row_positions)}
    fold_slices = [
        np.asarray([pos_map[r] for r in item.fold.val_idx if r in pos_map], dtype=int)
        for item in prepared
    ]

    raw = pd.DataFrame({
        family: family_results[family].raw_predictions[mask]
        for family in FAMILIES
    })
    rows = frame.iloc[row_positions].reset_index(drop=True)
    y = rows[TARGET].to_numpy(dtype=float)
    return raw, rows, y, fold_slices


def final_iteration_count(result, full_train_rows: int, parameter_ceiling: int) -> int:
    if not result.best_iterations:
        return int(parameter_ceiling)

    typical = float(np.median(result.best_iterations))
    inner_train = (
        float(np.mean(result.train_sizes))
        if result.train_sizes
        else float(full_train_rows)
    )
    scale = full_train_rows / inner_train if inner_train > 0 else 1.0
    return max(1, min(int(parameter_ceiling), int(round(typical * scale))))

In [14]:
def fit_final_family_models(
    train: pd.DataFrame,
    future: pd.DataFrame,
    features: list[str],
    best: dict[str, dict[str, Any]],
    family_results: dict[str, Any],
    catboost_features: int,
    jobs: int,
    selector_iterations: int = 300,
):
    """Refit all three families using training-only preprocessing/selection."""
    imputer = ForwardMedianImputer(features)
    train_i, future_i = imputer.fit_transform_pair(train, future)

    cat_features = fit_catboost_selector(
        train_i,
        features,
        feature_count=catboost_features,
        jobs=jobs,
        iterations=selector_iterations,
    )

    predictions = {}

    lp = dict(best["lgbm"])
    lgb_ceiling = int(lp["n_estimators"])
    lp["n_estimators"] = final_iteration_count(
        family_results["lgbm"], len(train_i), lgb_ceiling
    )
    lgb_model = lgb.LGBMRegressor(
        **lp,
        random_state=SEED,
        verbosity=-1,
        n_jobs=jobs,
    )
    lgb_model.fit(train_i[features], train_i[TARGET])
    predictions["lgbm"] = lgb_model.predict(future_i[features])

    cp = dict(best["catboost"])
    cat_ceiling = int(cp["iterations"])
    cp["iterations"] = final_iteration_count(
        family_results["catboost"], len(train_i), cat_ceiling
    )
    cp.update({
        "random_seed": SEED,
        "verbose": False,
        "loss_function": "RMSE",
        "task_type": "CPU",
        "thread_count": jobs,
        "boosting_type": "Plain",
    })
    cat_model = CatBoostRegressor(**cp)
    cat_model.fit(train_i[cat_features], train_i[TARGET])
    predictions["catboost"] = cat_model.predict(future_i[cat_features])

    scaler = StandardScaler().fit(train_i[features])
    ridge_model = Ridge(alpha=float(best["ridge"]["alpha"]))
    ridge_model.fit(scaler.transform(train_i[features]), train_i[TARGET])
    predictions["ridge"] = ridge_model.predict(scaler.transform(future_i[features]))

    return predictions, cat_features

In [15]:
def fit_inner_pipeline(train: pd.DataFrame, features: list[str], cfg: RunConfig):
    # FIX 1 + FIX 2: every inner fold owns its imputer and CatBoost selector.
    prepared = prepare_folds(
        train,
        features,
        n_splits=cfg.inner_splits,
        feature_count=cfg.catboost_features,
        jobs=cfg.jobs,
        selector_iterations=cfg.selector_iterations,
    )

    trial_counts = {
        "lgbm": cfg.trials_lgbm,
        "catboost": cfg.trials_catboost,
        "ridge": cfg.trials_ridge,
    }

    # FIX 3: model parameters are tuned on inner CV only.
    best = {}
    for family in FAMILIES:
        best[family], _ = tune_family(
            family,
            prepared,
            features,
            cfg.jobs,
            len(train),
            trial_counts[family],
        )

    family_results = {
        family: family_oof_predictions(
            prepared,
            family,
            best[family],
            features,
            cfg.jobs,
            len(train),
        )
        for family in FAMILIES
    }

    raw_oof, oof_rows, y_oof, fold_slices = compact_oof(
        train, prepared, family_results
    )

    # Standardisation statistics are learned from inner OOF predictions only.
    scaled_oof, family_stats = standardize_family_oof(raw_oof)

    # FIX 4: blend weights are tuned only on inner OOF predictions.
    weights, _ = tune_blend_weights(
        scaled_oof,
        y_oof,
        fold_slices,
        n_trials=cfg.trials_blend,
    )

    blended_scaled = blend(scaled_oof, weights)
    blend_stats = standardization_stats(blended_scaled)
    trading_signal = apply_standardization(blended_scaled, blend_stats)

    # FIX 6: calibrate risk using inner OOF only. This is a volatility
    # constraint, not direct Sharpe maximisation.
    risk_scale = calibrate_vol_budget_scale(
        trading_signal,
        oof_rows["vol_20"].to_numpy(dtype=float),
        oof_rows["forward_returns"].to_numpy(dtype=float),
        oof_rows["risk_free_rate"].to_numpy(dtype=float),
        target_vol_ratio=cfg.target_vol_ratio,
        target_vol=cfg.target_vol,
    )

    # FIX 5: forecast metrics stay in target units. Trading signal is separate.
    raw_forecast = blend(raw_oof, weights)
    oof_positions = risk_controlled_vol_budget_allocation(
        trading_signal,
        oof_rows["vol_20"].to_numpy(dtype=float),
        tilt_scale=risk_scale,
        target_vol=cfg.target_vol,
    )
    oof_metrics = evaluate(
        y_oof,
        raw_forecast,
        weights=oof_positions,
        forward_returns=oof_rows["forward_returns"].to_numpy(dtype=float),
        risk_free_rate=oof_rows["risk_free_rate"].to_numpy(dtype=float),
    )

    return {
        "best": best,
        "family_results": family_results,
        "family_stats": family_stats,
        "weights": weights,
        "blend_stats": blend_stats,
        "risk_scale": risk_scale,
        "inner_oof_metrics": oof_metrics,
    }

## 4. Outer-fold evaluation

Nothing in an outer validation fold may influence preprocessing, feature selection, model parameters, blend weights, or risk calibration.

In [16]:
def evaluate_outer_fold(frame, fold, features, cfg):
    outer_train = frame.iloc[fold.train_idx].copy()
    outer_val = frame.iloc[fold.val_idx].copy()

    inner = fit_inner_pipeline(outer_train, features, cfg)

    raw_pred_dict, cat_features = fit_final_family_models(
        outer_train,
        outer_val,
        features,
        inner["best"],
        inner["family_results"],
        cfg.catboost_features,
        cfg.jobs,
        cfg.selector_iterations,
    )
    raw_pred = pd.DataFrame(raw_pred_dict)

    scaled = pd.DataFrame({
        family: apply_standardization(
            raw_pred[family].to_numpy(),
            inner["family_stats"][family],
        )
        for family in FAMILIES
    })

    blended = blend(scaled, inner["weights"])
    trading_signal = causal_standardise(
        blended,
        warmup_mean=inner["blend_stats"][0],
        warmup_sd=inner["blend_stats"][1],
        min_periods=20,
    )

    # Forecast in target units for R²/RMSE/Spearman.
    raw_forecast = blend(raw_pred, inner["weights"])

    positions = risk_controlled_vol_budget_allocation(
        trading_signal,
        outer_val["vol_20"].to_numpy(dtype=float),
        tilt_scale=inner["risk_scale"],
        target_vol=cfg.target_vol,
    )

    metrics = evaluate(
        outer_val[TARGET].to_numpy(dtype=float),
        raw_forecast,
        weights=positions,
        forward_returns=outer_val["forward_returns"].to_numpy(dtype=float),
        risk_free_rate=outer_val["risk_free_rate"].to_numpy(dtype=float),
    )
    metrics.update({
        "fold": int(fold.index),
        "signal_spearman_ic": spearman_ic(
            outer_val[TARGET].to_numpy(dtype=float), trading_signal
        ),
        "risk_scale": float(inner["risk_scale"]),
        "n_cat_features": len(cat_features),
        **{
            f"weight_{family}": float(weight)
            for family, weight in inner["weights"].items()
        },
    })

    predictions = pd.DataFrame({
        DATE_COL: outer_val[DATE_COL].to_numpy(),
        "target": outer_val[TARGET].to_numpy(dtype=float),
        "forecast": raw_forecast,
        "trading_signal": trading_signal,
        "allocation": positions,
        **{
            f"pred_{family}": raw_pred[family].to_numpy()
            for family in FAMILIES
        },
    })

    audit = {
        "best_params": inner["best"],
        "weights": inner["weights"],
        "risk_scale": inner["risk_scale"],
        "inner_oof_metrics": inner["inner_oof_metrics"],
        "catboost_features": cat_features,
    }
    return metrics, predictions, audit

## 5. Run nested outer CV

For a real comparison with the previous ensemble, set `QUICK = False` in the configuration cell and restart/run all cells. The quick mode is only a technical smoke test.

In [17]:
def _json_default(value):
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(type(value).__name__)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
outer_folds = get_folds(train_df, n_splits=CFG.outer_splits)

fold_metrics = []
all_predictions = []
audits = {}
started = time.time()

print(f"Outer folds: {len(outer_folds)} | inner folds: {CFG.inner_splits}")
print("Primary estimate: nested outer-CV Modified Sharpe")

for fold in outer_folds:
    print(f"\n=== outer fold {fold.index + 1}/{len(outer_folds)} ===")
    metrics, predictions, audit = evaluate_outer_fold(
        train_df, fold, FEATURES, CFG
    )

    fold_metrics.append(metrics)
    predictions.insert(0, "fold", fold.index)
    all_predictions.append(predictions)
    audits[str(fold.index)] = audit

    print(
        "modified_sharpe={modified_sharpe:+.4f} "
        "raw_sharpe={sharpe:+.4f} "
        "spearman={spearman_ic:+.4f} "
        "r2={r2:+.4f} rmse={rmse:.6f} "
        "vol_ratio={vol_ratio:.3f}".format(**metrics)
    )
    print(
        "weights=" + str({
            family: round(metrics[f"weight_{family}"], 3)
            for family in FAMILIES
        }) + f" risk_scale={metrics['risk_scale']:.3f}"
    )

fold_df = pd.DataFrame(fold_metrics)
pred_df = pd.concat(all_predictions, ignore_index=True)
summary = aggregate_folds([
    {k: v for k, v in row.items() if k != "fold"}
    for row in fold_metrics
])
summary["elapsed_seconds"] = time.time() - started
summary["method"] = "nested_outer_cv"
summary["config"] = asdict(CFG)
summary["n_features"] = len(FEATURES)

fold_df.to_csv(OUTPUT_DIR / "outer_folds.csv", index=False)
pred_df.to_csv(OUTPUT_DIR / "outer_predictions.csv", index=False)
(OUTPUT_DIR / "outer_audit.json").write_text(
    json.dumps(audits, indent=2, default=_json_default)
)
(OUTPUT_DIR / "summary.json").write_text(
    json.dumps(summary, indent=2, default=_json_default)
)

print("\n=== nested CV summary ===")
for key in (
    "modified_sharpe_mean",
    "modified_sharpe_std",
    "sharpe_mean",
    "spearman_ic_mean",
    "r2_mean",
    "rmse_mean",
    "vol_ratio_mean",
):
    if key in summary:
        print(f"{key}: {summary[key]:+.6f}")

fold_df

Outer folds: 4 | inner folds: 3
Primary estimate: nested outer-CV Modified Sharpe

=== outer fold 1/4 ===


/home/alex/Desktop/PMDL-project/PMDL-Project-Market-Prediction/vr/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/home/alex/Desktop/PMDL-project/PMDL-Project-Market-Prediction/vr/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/home/alex/Desktop/PMDL-project/PMDL-Project-Market-Prediction/vr/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/home/alex/Desktop/PMDL-project/PMDL-Project-Market-Prediction/vr/lib/python3.12/site-packa

modified_sharpe=-0.1151 raw_sharpe=-0.1151 spearman=+0.0494 r2=-0.0079 rmse=0.011386 vol_ratio=1.031
weights={'lgbm': 0.382, 'catboost': 0.528, 'ridge': 0.09} risk_scale=0.325

=== outer fold 2/4 ===


/home/alex/Desktop/PMDL-project/PMDL-Project-Market-Prediction/vr/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/home/alex/Desktop/PMDL-project/PMDL-Project-Market-Prediction/vr/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/home/alex/Desktop/PMDL-project/PMDL-Project-Market-Prediction/vr/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/home/alex/Desktop/PMDL-project/PMDL-Project-Market-Prediction/vr/lib/python3.12/site-packa

modified_sharpe=+0.2654 raw_sharpe=+0.2654 spearman=+0.0406 r2=-0.0025 rmse=0.013207 vol_ratio=1.052
weights={'lgbm': 0.34, 'catboost': 0.636, 'ridge': 0.024} risk_scale=0.250

=== outer fold 3/4 ===


/home/alex/Desktop/PMDL-project/PMDL-Project-Market-Prediction/vr/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/home/alex/Desktop/PMDL-project/PMDL-Project-Market-Prediction/vr/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/home/alex/Desktop/PMDL-project/PMDL-Project-Market-Prediction/vr/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/home/alex/Desktop/PMDL-project/PMDL-Project-Market-Prediction/vr/lib/python3.12/site-packa

modified_sharpe=+0.8120 raw_sharpe=+0.8159 spearman=-0.0038 r2=-0.0078 rmse=0.007943 vol_ratio=1.062
weights={'lgbm': 0.569, 'catboost': 0.378, 'ridge': 0.053} risk_scale=0.250

=== outer fold 4/4 ===


/home/alex/Desktop/PMDL-project/PMDL-Project-Market-Prediction/vr/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/home/alex/Desktop/PMDL-project/PMDL-Project-Market-Prediction/vr/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/home/alex/Desktop/PMDL-project/PMDL-Project-Market-Prediction/vr/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/home/alex/Desktop/PMDL-project/PMDL-Project-Market-Prediction/vr/lib/python3.12/site-packa

modified_sharpe=+0.7434 raw_sharpe=+0.7434 spearman=+0.0670 r2=+0.0020 rmse=0.011050 vol_ratio=1.035
weights={'lgbm': 0.017, 'catboost': 0.79, 'ridge': 0.193} risk_scale=0.175

=== nested CV summary ===
modified_sharpe_mean: +0.426413
modified_sharpe_std: +0.376942
sharpe_mean: +0.427394
spearman_ic_mean: +0.038310
r2_mean: -0.004064
rmse_mean: +0.010896
vol_ratio_mean: +1.044913


,rmse,r2,spearman_ic,hit_rate,n_rows,modified_sharpe,sharpe,vol_ratio,vol_penalty,return_penalty,...,benchmark_sharpe,mean_weight,weight_turnover,fold,signal_spearman_ic,risk_scale,n_cat_features,weight_lgbm,weight_catboost,weight_ridge
0,0.011386,-0.007927,0.049440,0.527062,1552,-0.115115,-0.115115,1.031275,1.0,1.000000,...,-0.151346,0.928101,0.120772,0,0.047382,0.325,120,0.381673,0.528144,0.090183
1,0.013207,-0.002528,0.040610,0.496134,1552,0.265369,0.265369,1.051751,1.0,1.000000,...,0.356243,0.964730,0.074289,1,0.037813,0.250,120,0.340418,0.635852,0.023730
2,0.007943,-0.007817,-0.003804,0.481314,1552,0.811953,0.815875,1.061588,1.0,1.004831,...,0.984320,0.964999,0.111067,2,-0.008432,0.250,120,0.569210,0.377862,0.052928
3,0.011050,0.002016,0.066993,0.500000,1554,0.743447,0.743447,1.035038,1.0,1.000000,...,0.819368,0.976029,0.041382,3,0.060905,0.175,120,0.016903,0.789736,0.193361


## 6. Optional diagnostic evaluation on the final 180 rows

**Do not use this block to choose parameters.** Earlier team iterations have already inspected this period, so it is not an untouched test set anymore. Keep `EVALUATE_DIAGNOSTIC_PUBLIC=False` during model development.

In [18]:
def evaluate_diagnostic_public(train, public, features, cfg):
    inner = fit_inner_pipeline(train, features, cfg)
    raw_pred_dict, cat_features = fit_final_family_models(
        train,
        public,
        features,
        inner["best"],
        inner["family_results"],
        cfg.catboost_features,
        cfg.jobs,
        cfg.selector_iterations,
    )
    raw_pred = pd.DataFrame(raw_pred_dict)

    scaled = pd.DataFrame({
        family: apply_standardization(
            raw_pred[family].to_numpy(), inner["family_stats"][family]
        )
        for family in FAMILIES
    })
    blended = blend(scaled, inner["weights"])
    trading_signal = causal_standardise(
        blended,
        warmup_mean=inner["blend_stats"][0],
        warmup_sd=inner["blend_stats"][1],
        min_periods=20,
    )
    raw_forecast = blend(raw_pred, inner["weights"])
    positions = risk_controlled_vol_budget_allocation(
        trading_signal,
        public["vol_20"].to_numpy(dtype=float),
        tilt_scale=inner["risk_scale"],
        target_vol=cfg.target_vol,
    )

    metrics = evaluate(
        public[TARGET].to_numpy(dtype=float),
        raw_forecast,
        weights=positions,
        forward_returns=public["forward_returns"].to_numpy(dtype=float),
        risk_free_rate=public["risk_free_rate"].to_numpy(dtype=float),
    )
    predictions = pd.DataFrame({
        DATE_COL: public[DATE_COL].to_numpy(),
        "target": public[TARGET].to_numpy(dtype=float),
        "forecast": raw_forecast,
        "trading_signal": trading_signal,
        "allocation": positions,
        **{
            f"pred_{family}": raw_pred[family].to_numpy()
            for family in FAMILIES
        },
    })
    return metrics, predictions, {
        "best_params": inner["best"],
        "weights": inner["weights"],
        "risk_scale": inner["risk_scale"],
        "catboost_features": cat_features,
    }

if EVALUATE_DIAGNOSTIC_PUBLIC:
    diagnostic_metrics, diagnostic_predictions, diagnostic_audit = evaluate_diagnostic_public(
        train_df, public_df, FEATURES, CFG
    )
    diagnostic_predictions.to_csv(
        OUTPUT_DIR / "diagnostic_public_predictions.csv", index=False
    )
    (OUTPUT_DIR / "diagnostic_public.json").write_text(
        json.dumps(
            {"metrics": diagnostic_metrics, "audit": diagnostic_audit},
            indent=2,
            default=_json_default,
        )
    )
    print("DIAGNOSTIC ONLY:")
    print(diagnostic_metrics)
else:
    print("Diagnostic public evaluation disabled (recommended during development).")

Diagnostic public evaluation disabled (recommended during development).


## 7. Interpretation

For the final comparison, use the full run (`QUICK=False`) and report the outer-CV mean and standard deviation. Do not compare the quick smoke result directly with the previous 4-fold full run.

The most important outputs are:

- `modified_sharpe_mean` — primary competition-style performance estimate;
- `modified_sharpe_std` — regime stability;
- `vol_ratio_mean` — whether the strategy stays below the volatility penalty region;
- `spearman_ic_mean` — ranking quality of the raw return forecast;
- `rmse_mean` and `r2_mean` — regression diagnostics computed on forecasts in target units.